In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from flaml import AutoML
from sklearn.metrics import mean_squared_error

In [12]:
df = pd.read_csv('final.csv')

numeric_data = df.select_dtypes(include=['number'])

numeric_data = numeric_data.rename(columns=lambda x: x.replace('[', '').replace(']', '').replace('<', '').replace(' ', '_'))

train_size = int(0.8 * len(numeric_data))
train_data = numeric_data[:train_size]
test_data = numeric_data[train_size:]

target_column = 'Day-ahead_prices_Germany/Luxembourg_€/MWh_Original_resolutions'

X_train = train_data.drop(columns=[target_column])
y_train = train_data[target_column]

X_test = test_data.drop(columns=[target_column])
y_test = test_data[target_column]

In [ ]:
automl = AutoML()

automl_settings = {
    "time_budget": 180,  
    "metric": "rmse",
    "task": "regression",  
    "log_file_name": "automl.log",  
    "estimator_list": ["catboost", "lgbm", "rf", "xgboost"], 
    "n_jobs": -1  
}

automl.fit(X_train, y_train, **automl_settings)

print("Best Model:", automl.model)  
print("Best Model Hyperparameters:", automl.best_config) 

predictions = automl.predict(X_test)

mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
print(f'RMSE: {rmse}')